# EZNX-ATLAS-A Colab Smoke Test

This notebook is for a fresh Google Colab check of the published reproducibility surface.

- `infra` mode: install + published checksum verification only.
- `both` mode: one initial run + one complementary run, then normalized JSON comparison against the archived references.

Recommended runtime: **CPU**.

In [ ]:
REPO_URL = "https://github.com/ezynsegnane/run_github_colab.git"
REPO_DIR = "/content/run_github_colab"
RUN_MODE = "infra"  # change to "both" for the full smoke test

print("REPO_URL =", REPO_URL)
print("REPO_DIR =", REPO_DIR)
print("RUN_MODE =", RUN_MODE)

In [ ]:
import os
import shutil
import subprocess

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Current working directory:", os.getcwd())

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.3.1+cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "wfdb"], check=True)

In [ ]:
from pathlib import Path
import wfdb

def find_ptbxl_root(search_root: str) -> str | None:
    for path in Path(search_root).rglob("ptbxl_database.csv"):
        return str(path.parent)
    return None

DATA_ROOT = find_ptbxl_root("/content")
if DATA_ROOT is None:
    wfdb.dl_database("ptb-xl", "/content/ptb-xl")
    DATA_ROOT = find_ptbxl_root("/content")

if DATA_ROOT is None:
    raise FileNotFoundError("Could not locate the PTB-XL root after download.")

print("DATA_ROOT =", DATA_ROOT)

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "reproducibility/verify_reproducibility.py", "--mode", "both", "--verify-published"],
    check=True,
)

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "reproducibility/run_colab_smoke_test.py",
        "--mode",
        RUN_MODE,
        "--data-root",
        DATA_ROOT,
    ],
    check=True,
)